# Navitas — v2 Model VAL Testi (Colab) · Sürüm 2

**Ne yapar:** Sahadaki modeli (`models/navitas_seg_v2_best.pt`) eğitimdeki doğrulama
setinin birebir aynısında ölçer; **sınıf bazlı** (traversable/blocked) mAP çıkarır.

**Çıktı:** `navitas_v2_val_<tarih>.zip` — tüm loglar + sonuçlar. İndirip Claude'a ver.

**Sıralama mantığı:** önce ucuz ve kritik işler (model var mı? kimliği ne?),
EN SONA uzun veri indirme + ölçüm. Böylece bir eksik varsa 1. dakikada patlar,
21. dakikada değil.

**Başlamadan:** `Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU`.
Her hücrenin sonunda `✓ ADIM n TAMAM` yazısını gör, sonra devam et.

## Adım 1 — Ortam + log dosyası
Tüm çıktılar ekranda NORMAL görünür; ek olarak `konsol.log`'a da yazılır.

In [ ]:
import os, sys, json, subprocess, datetime, platform

CALISMA = "/content/navitas_val_testi"
os.makedirs(CALISMA, exist_ok=True)
LOG = f"{CALISMA}/konsol.log"

def logla(*par):
    """print gibi: ekrana basar + log dosyasına ekler."""
    metin = " ".join(str(p) for p in par)
    print(metin)
    with open(LOG, "a", encoding="utf-8") as f:
        f.write(metin + "\n")

def komut(cmd, gizle=None):
    """Kabuk komutu: çıktı satır satır EKRANDA + log'da."""
    goster = cmd.replace(gizle, "***") if gizle else cmd
    logla(f"\n$ {goster}")
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, errors="replace")
    for satir in p.stdout:
        # tqdm'in \r ile ezdiği ilerleme satırlarından son hâli al
        logla(satir.rstrip("\n").split("\r")[-1])
    p.wait()
    logla(f"[çıkış kodu: {p.returncode}]")
    return p.returncode

logla("=" * 60)
logla("NAVITAS v2 VAL TESTİ —", datetime.datetime.now().isoformat())
logla("=" * 60)
komut("nvidia-smi | head -12 || echo 'GPU YOK — CPU ile de çalışır, sadece yavaş'")
komut("pip install -q ultralytics datasets pillow tqdm opencv-python-headless")

import torch, ultralytics
ortam = {
    "tarih": datetime.datetime.now().isoformat(),
    "python": platform.python_version(),
    "torch": torch.__version__,
    "ultralytics": ultralytics.__version__,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "yok (CPU)",
}
json.dump(ortam, open(f"{CALISMA}/ortam.json", "w"), indent=2, ensure_ascii=False)
logla("Ortam:", json.dumps(ortam, ensure_ascii=False))
logla("\n✓ ADIM 1 TAMAM")

## Adım 2 — Repoyu indir
Token'ı sorunca yapıştır (ekranda ve logda görünmez). Repo zaten inikse atlar.

In [ ]:
from getpass import getpass

if not os.path.isdir("/content/navitas/.git"):
    token = getpass("GitHub token (repo herkese açıksa boş bırak + Enter): ").strip()
    url = (f"https://{token}@github.com/navitas26/navitas.git" if token
           else "https://github.com/navitas26/navitas.git")
    rc = komut(f"git clone --depth 1 {url} /content/navitas", gizle=token or None)
    if rc != 0:
        raise SystemExit("Klonlama BAŞARISIZ — token'ı/interneti kontrol et, hücreyi tekrar çalıştır")
else:
    logla("Repo zaten var, klonlama atlandı.")

komut("ls -la /content/navitas/models/")

# PUSH DOĞRULAMA — spam bayraklı hesapta push'un gerçekten ulaştığının kanıtı
ISARET_BEKLENEN = "NAVITAS-PUSH-20260711-beea23"
isaret_yolu = "/content/navitas/push_isareti.txt"
if os.path.isfile(isaret_yolu):
    okunan = open(isaret_yolu).read().strip()
    if okunan == ISARET_BEKLENEN:
        logla(f"\n✅ PUSH DOĞRULANDI — işaret eşleşti: {okunan}")
    else:
        logla(f"\n⚠ İşaret dosyası var ama İÇERİK FARKLI: {okunan} (beklenen: {ISARET_BEKLENEN}) — eski bir push görüyorsun olabilir")
else:
    logla("\n❌ push_isareti.txt YOK — son push repoya ULAŞMAMIŞ (ya da eski klon önbelleği); Claude'a söyle")

logla("\n✓ ADIM 2 TAMAM")

## Adım 3 — Model kontrolü + kimlik (veri indirmeden ÖNCE)
Doğru dosya: `models/navitas_seg_v2_best.pt`. Yoksa sana yükletir. SHA-256 + gömülü eğitim bilgileri kayda geçer.

In [ ]:
import hashlib, torch

MODEL_PT = "/content/navitas/models/navitas_seg_v2_best.pt"

if not os.path.isfile(MODEL_PT):
    logla("⚠ Model repoda bulunamadı — bilgisayarından yükle (navitas_seg_v2_best.pt):")
    from google.colab import files
    yuklenen = files.upload()
    ad = next(iter(yuklenen))
    os.makedirs(os.path.dirname(MODEL_PT), exist_ok=True)
    os.replace(f"/content/{ad}", MODEL_PT)

boyut = os.path.getsize(MODEL_PT)
sha = hashlib.sha256(open(MODEL_PT, "rb").read()).hexdigest()
logla(f"Model: {MODEL_PT}")
logla(f"Boyut: {boyut/1e6:.1f} MB · SHA-256: {sha}")
assert boyut > 1_000_000, "Dosya şüpheli derecede küçük — yanlış dosya olabilir"

ckpt = torch.load(MODEL_PT, map_location="cpu", weights_only=False)
kimlik = {
    "dosya": MODEL_PT, "sha256": sha, "boyut_bayt": boyut,
    "egitim_tarihi": str(ckpt.get("date")),
    "ultralytics_surumu": str(ckpt.get("version")),
    "gomulu_val_metrikleri": ckpt.get("train_metrics"),
    "egitim_ayarlari": {k: ckpt["train_args"].get(k) for k in
        ("data","epochs","imgsz","batch","optimizer","lr0","lrf","patience","cos_lr","seed")},
}
json.dump(kimlik, open(f"{CALISMA}/model_kimlik.json", "w"),
          indent=2, ensure_ascii=False, default=str)
logla(json.dumps(kimlik, indent=2, ensure_ascii=False, default=str))
del ckpt
logla("\n✓ ADIM 3 TAMAM — model doğrulandı, artık uzun işe geçebiliriz")

## Adım 4 — Veri setlerini indir (EN UZUN adım, ~10–20 dk ☕)
Eğitimdekiyle aynı: RescueNet + FloodNet. İlerleme satırları ekranda akacak.

In [ ]:
os.chdir("/content/navitas/dataset")
logla("=== RescueNet indiriliyor/dönüştürülüyor ===")
assert komut("python3 download_and_convert.py") == 0, "RescueNet indirme başarısız"
logla("=== FloodNet indiriliyor/dönüştürülüyor ===")
assert komut("python3 download_floodnet.py") == 0, "FloodNet indirme başarısız"
logla("\n✓ ADIM 4 TAMAM — veri setleri hazır")

## Adım 5 — Eğitimdeki ayar dosyasının birebir aynısı + sayım
v2 eğitiminde val = `rescuenet_yolo/val` + `floodnet_yolo/val` idi; aynısını kuruyoruz.

In [ ]:
import glob

yaml_icerik = """train:
  - /content/navitas/dataset/rescuenet_yolo/images/train
  - /content/navitas/dataset/floodnet_yolo/images/train

val:
  - /content/navitas/dataset/rescuenet_yolo/images/val
  - /content/navitas/dataset/floodnet_yolo/images/val

test: /content/navitas/dataset/rescuenet_yolo/images/test

nc: 2
names:
  0: traversable
  1: blocked
"""
VERI_YAML = "/content/navitas/dataset/combined_data_val.yaml"
open(VERI_YAML, "w").write(yaml_icerik)
logla(yaml_icerik)

sayim = {}
for kaynak in ("rescuenet_yolo", "floodnet_yolo"):
    n = len(glob.glob(f"/content/navitas/dataset/{kaynak}/images/val/*"))
    sayim[kaynak + "/val"] = n
    logla(f"{kaynak}/images/val: {n} görsel")
for kaynak, n in sayim.items():
    if n == 0:
        logla(f"🔴 UYARI: {kaynak} BOŞ (0 görsel) — bu kaynak ölçüme GİRMEYECEK!")
        logla("   (Bilinen durum: FloodNet HuggingFace kaynağı 11.07.2026'dan beri görüntü vermiyor.)")
assert sum(sayim.values()) > 0, "Hiç val görseli yok — Adım 4 gerçekten bitti mi?"
json.dump(sayim, open(f"{CALISMA}/dataset_ozet.json", "w"), indent=2)
logla("\n✓ ADIM 5 TAMAM")

## Adım 6 — VAL ölçümü (asıl test, ~2–5 dk)
Ekranda Ultralytics'in kendi tablosu; ayrıca sınıf bazlı sonuç JSON'a yazılır.

In [ ]:
import logging
from ultralytics import YOLO

# Ultralytics'in kendi loglarını da konsol.log'a düşür (ekran aynen kalır)
fh = logging.FileHandler(LOG)
fh.setFormatter(logging.Formatter("%(message)s"))
logging.getLogger("ultralytics").addHandler(fh)

model = YOLO(MODEL_PT)
sonuc = model.val(data=VERI_YAML, imgsz=512, split="val", plots=True,
                  project=CALISMA, name="val_ciktilari", exist_ok=True)

metrikler = {
    "genel": {
        "box_mAP50": round(sonuc.box.map50, 4),
        "box_mAP50_95": round(sonuc.box.map, 4),
        "mask_mAP50": round(sonuc.seg.map50, 4),
        "mask_mAP50_95": round(sonuc.seg.map, 4),
        "mask_precision": round(sonuc.seg.mp, 4),
        "mask_recall": round(sonuc.seg.mr, 4),
    },
    "sinif_bazli": {},
    "hiz_ms": sonuc.speed,
}
for sira, sinif_idx in enumerate(sonuc.ap_class_index):
    ad = sonuc.names[int(sinif_idx)]
    metrikler["sinif_bazli"][ad] = {
        "box_mAP50": round(float(sonuc.box.ap50[sira]), 4),
        "mask_mAP50": round(float(sonuc.seg.ap50[sira]), 4),
        "mask_mAP50_95": round(float(sonuc.seg.ap[sira]), 4),
    }
json.dump(metrikler, open(f"{CALISMA}/val_sonuclari.json", "w"), indent=2, ensure_ascii=False)

logla("\n" + "=" * 60)
logla("SONUÇLAR")
logla(json.dumps(metrikler, indent=2, ensure_ascii=False))
m = metrikler["genel"]["mask_mAP50"]
gomulu = (kimlik["gomulu_val_metrikleri"] or {}).get("metrics/mAP50(M)")
logla(f"\nGömülü eğitim-sonu değeri: {gomulu} · Bu koşu: {m} (yakınsa ölçüm sağlıklı)")
logla(f"Takım hedefi mask mAP50 ≥ 0,60 → ölçülen {m} →",
      "KARŞILANDI ✓" if m >= 0.60 else "KARŞILANMADI ✗ (v3 iyileştirme gerekçesi)")
logla("\n✓ ADIM 6 TAMAM")

## Adım 7 — Özet + zip + indir
Bir şey ters gitmiş olsa bile bu hücre çalışır ve eldekini paketler.

In [ ]:
import shutil

try:
    m
except NameError:
    m, metrikler = None, {"genel": {}, "sinif_bazli": {}}

ozet = f"""# Navitas v2 VAL Testi — Özet
Tarih: {datetime.datetime.now().isoformat()}
Model SHA-256: {sha if 'sha' in dir() else '?'}
Ortam: {ortam.get('gpu')} · ultralytics {ortam.get('ultralytics')} · torch {ortam.get('torch')}

Genel : {json.dumps(metrikler.get('genel', {}), ensure_ascii=False)}
Sınıf : {json.dumps(metrikler.get('sinif_bazli', {}), ensure_ascii=False)}

Dosyalar: konsol.log · ortam.json · model_kimlik.json · val_sonuclari.json ·
dataset_ozet.json · val_ciktilari/ (confusion matrix, PR eğrileri, örnek kareler)
"""
open(f"{CALISMA}/00_OZET.md", "w").write(ozet)
logla(ozet)

zaman = datetime.datetime.now().strftime("%Y%m%d_%H%M")
zip_adi = f"/content/navitas_v2_val_{zaman}"
shutil.make_archive(zip_adi, "zip", CALISMA)
logla("Zip hazır:", zip_adi + ".zip")

from google.colab import files
files.download(zip_adi + ".zip")

---
**Bitince:** inen `navitas_v2_val_*.zip`'i repo köküne koy, Claude'a haber ver → v4.2'ye işlenecek.
**Sorun çıkarsa:** hiçbir şeyi silme; Adım 7'yi yine de çalıştır (eldekini paketler) ve zip'i paylaş.